<a href="https://colab.research.google.com/github/sachinthadilshann/pytorch_course_by_DanielBourke/blob/main/5_pytorch_going_modular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import os
import requests
import zipfile
from pathlib import Path


data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)


with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print("Downloading pizza, steak, sushi data...")
    f.write(request.content)


with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping pizza, steak, sushi data...")
    zip_ref.extractall(image_path)


os.remove(data_path / "pizza_steak_sushi.zip")

data/pizza_steak_sushi directory exists.
Unzipping pizza, steak, sushi data...


In [16]:
train_dir = image_path / "train"
test_dir  = image_path / "test"



In [17]:
train_dir

PosixPath('data/pizza_steak_sushi/train')

##Create Datasets and DataLoaders (data_setup.py)

In [18]:
import os

os.makedirs("going_modular",exist_ok=True)

In [19]:
%%writefile going_modular/data_setup.py

import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


NUM_WORKERS = os.cpu_count()


def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int = NUM_WORKERS
):
    train_data = datasets.ImageFolder(
        train_dir,
        transform=transform
    )

    test_data = datasets.ImageFolder(
        test_dir,
        transform=transform
    )

    class_names = train_data.classes

    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )

    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_dataloader, test_dataloader, class_names

Overwriting going_modular/data_setup.py


##Making Model

In [20]:
%%writefile going_modular/model_builder.py

import torch
from torch import nn


class TinyVGG(nn.Module):
  def __init__(self,
               input_shape:int,
               hidden_units:int,
               output_shape:int) -> None:
      super().__init__()
      self.conv_block_1 = nn.Sequential(
          nn.Conv2d(in_channels=input_shape,
                    out_channels=hidden_units,
                    kernel_size = 3,
                    stride=1,
                    padding=0),
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=0),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2,
                       stride=2)
      )

      self.conv_block_2 = nn.Sequential(
          nn.Conv2d(hidden_units,
                    hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=0
                    ),
          nn.ReLU(),
          nn.Conv2d(hidden_units,hidden_units,kernel_size=3,stride=1,padding=0),
          nn.ReLU(),
          nn.MaxPool2d(2)
      )

      self.classifier = nn.Sequential(
          nn.Flatten(),
          nn.Linear(in_features=hidden_units*13*13,
                    out_features=output_shape)
      )
  def forward(self,x:torch.Tensor):
    return self.classifier(self.conv_block_2(self.conv_block_1(x)))

Overwriting going_modular/model_builder.py


In [21]:
'''import torch
import importlib
from torchvision import transforms
from going_modular import data_setup,model_builder


device = "cuda" if torch.cuda.is_available() else "cpu"


model = model_builder.TinyVGG(input_shape=3,
                              hidden_units=10,
                              output_shape=len(class_names)).to(device)
model '''

'import torch\nimport importlib\nfrom torchvision import transforms\nfrom going_modular import data_setup,model_builder\n\n\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\n\n\nmodel = model_builder.TinyVGG(input_shape=3,\n                              hidden_units=10,\n                              output_shape=len(class_names)).to(device)\nmodel '

##Creating train_step() and test_step() functions and train() to combine them

In [22]:
%%writefile going_modular/engine.py

import torch

from tqdm.auto import tqdm
from typing import Dict,List,Tuple

def train_step(model:torch.nn.Module,
               dataloader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               optimizer:torch.optim.Optimizer,
               device:torch.device):

  model.train()
  train_loss,train_acc = 0,0

  for X,y in dataloader:
    X,y= X.to(device), y.to(device)

    y_pred = model(X)

    loss = loss_fn(y_pred,y)
    train_loss +=loss.item()

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    #cal_accuracy

    y_pred_class = y_pred.argmax(dim=1)
    train_acc += (y_pred_class == y).float().mean().item()

  train_loss = train_loss / len(dataloader)
  train_acc =  train_acc / len(dataloader)

  return train_loss,train_acc

def test_step(model:torch.nn.Module,
              dataloader:torch.utils.data.DataLoader,
              loss_fn:torch.nn.Module,
              device:torch.device):

  model.eval()

  test_loss,test_acc = 0,0

  with torch.inference_mode():
    for X,y in dataloader:
      X,y = X.to(device), y.to(device)
      test_pred_logits = model(X)

      loss = loss_fn(test_pred_logits,y)
      test_loss += loss.item()

      test_pred_labels = test_pred_logits.argmax(dim=1)
      test_acc += (test_pred_labels == y).float().mean().item()

  test_loss /= len(dataloader)
  test_acc /= len(dataloader)

  return test_loss,test_acc


def train(
    model:torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn:torch.nn.Module,
    epochs:int,
    device:torch.device,
):
  results = {"train_loss":[],
             "train_acc":[],
             "test_loss":[],
             "test_acc":[]}


  for epoch in tqdm(range(epochs)):
    train_loss,train_acc = train_step(model=model,
                                      dataloader=train_dataloader,
                                      optimizer=optimizer,
                                      loss_fn=loss_fn,
                                      device=device,
                                      )

    test_loss,test_acc = test_step(model=model,
                                    dataloader=test_dataloader,
                                    loss_fn=loss_fn,
                                    device=device,
                                   )

    print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  return results

Overwriting going_modular/engine.py


In [23]:
%%writefile going_modular/utils.py
"""
Contains various utility functions for PyTorch model training and saving.
"""
import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):

  # Create target directory
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True,
                        exist_ok=True)

  # Create model save path
  assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
  model_save_path = target_dir_path / model_name

  # Save the model state_dict()
  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(),
             f=model_save_path)



Overwriting going_modular/utils.py


In [24]:
%%writefile going_modular/train.py

import os
import torch
from torchvision import transforms
from going_modular import data_setup,engine,model_builder,utils

import importlib

importlib.reload(data_setup)
importlib.reload(engine)
importlib.reload(model_builder)
importlib.reload(utils)




NUM_EPOCHS = 5
BATCH_SIZE = 32
HIDDEN_UNITS = 10
LEARNING_RATE = 0.01

train_dir = "data/pizza_steak_sushi/train"
test_dir = "data/pizza_steak_sushi/test"

device = "cuda" if torch.cuda.is_available() else "cpu"

data_transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor()
])

train_dataloader,test_dataloader,class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE
)

model = model_builder.TinyVGG(
    input_shape=3,
    hidden_units=HIDDEN_UNITS,
    output_shape=len(class_names)
).to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE)

engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=NUM_EPOCHS,
             device=device)

utils.save_model(model=model,
                 target_dir="models",
                 model_name="05_going_modular_script_mode_tinyvgg_model.pth")



Overwriting going_modular/train.py


In [25]:
!ls /content/going_modular

data_setup.py  engine.py  model_builder.py  __pycache__  train.py  utils.py


In [26]:

!python -m going_modular.train

  0% 0/5 [00:00<?, ?it/s]Epoch: 1 | train_loss: 1.1289 | train_acc: 0.3086 | test_loss: 1.0940 | test_acc: 0.1979
 20% 1/5 [00:01<00:04,  1.17s/it]Epoch: 2 | train_loss: 1.1048 | train_acc: 0.2891 | test_loss: 1.0823 | test_acc: 0.5114
 40% 2/5 [00:02<00:03,  1.06s/it]Epoch: 3 | train_loss: 1.0961 | train_acc: 0.2891 | test_loss: 1.0991 | test_acc: 0.3191
 60% 3/5 [00:03<00:02,  1.31s/it]Epoch: 4 | train_loss: 1.0677 | train_acc: 0.3281 | test_loss: 1.0970 | test_acc: 0.3305
 80% 4/5 [00:05<00:01,  1.36s/it]Epoch: 5 | train_loss: 1.0969 | train_acc: 0.3164 | test_loss: 1.0932 | test_acc: 0.2604
100% 5/5 [00:06<00:00,  1.23s/it]
[INFO] Saving model to: models/05_going_modular_script_mode_tinyvgg_model.pth


In [28]:
!python -m going_modular.train --model model --batch_size 32 --lr 0.001 --num_epochs 5

  0% 0/5 [00:00<?, ?it/s]Epoch: 1 | train_loss: 1.1422 | train_acc: 0.3047 | test_loss: 1.0984 | test_acc: 0.2604
 20% 1/5 [00:01<00:04,  1.20s/it]Epoch: 2 | train_loss: 1.1006 | train_acc: 0.2891 | test_loss: 1.1006 | test_acc: 0.1979
 40% 2/5 [00:02<00:03,  1.07s/it]Epoch: 3 | train_loss: 1.1012 | train_acc: 0.2930 | test_loss: 1.0980 | test_acc: 0.1979
 60% 3/5 [00:03<00:02,  1.01s/it]Epoch: 4 | train_loss: 1.0993 | train_acc: 0.2891 | test_loss: 1.1000 | test_acc: 0.2604
 80% 4/5 [00:04<00:00,  1.02it/s]Epoch: 5 | train_loss: 1.0972 | train_acc: 0.3906 | test_loss: 1.1055 | test_acc: 0.1979
100% 5/5 [00:05<00:00,  1.05s/it]
[INFO] Saving model to: models/05_going_modular_script_mode_tinyvgg_model.pth
